In [2]:
# Restaurant Review Sentiment Analysis: NLP, Monitoring, and Streamlit Deployment

## Problem Statement

Restaurants receive customer feedback continuously through review platforms, delivery apps, surveys, and social media. Manually reading every review is slow, and simple average ratings may hide important customer experience issues.

The goal is to build a sentiment analysis system that classifies restaurant reviews as **positive** or **negative**, then prepares the solution for deployment using Streamlit. Along with the model, we will add practical monitoring so that a real user can track prediction quality signals, low-confidence reviews, input drift, and recent prediction behavior.

## Objectives
Load and understand the restaurant review dataset.
Preprocess review text for machine learning while preserving sentiment-important negation words such as not and never.
Train a text classification model using Bag of Words and Logistic Regression.
Evaluate model performance with useful classification metrics.
Build monitoring checks inside the notebook.
Create a Streamlit application for live prediction, batch scoring, and real-time monitoring.
End with a deployment-ready conclusion and next-step recommendations.

In [3]:
# Step 1: Import Required Libraries

import re 
import string
from collections import Counter # For counting word frequencies
from datetime import datetime # For timestamping model updates
from pathlib import Path # For handling file paths

import joblib # For saving and loading models
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer # For converting text to a matrix of token counts
from sklearn.linear_model import LogisticRegression # For building the sentiment analysis model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline # For creating a machine learning pipeline
# What is Pipeline? 
# A Pipeline in scikit-learn is a way to streamline a machine learning workflow by chaining together multiple steps, such as data preprocessing and model training, into a single object. This allows you to fit and predict with the entire workflow in one go, making it easier to manage and reproduce your machine learning processes.
# This is popularly used in Machine Learning projects to ensure that all steps are executed in the correct order and to simplify the code.
# All our earlier projects where not using Pipeline, but now we will be using it to make our code cleaner and more efficient.
# Earlier we were doing something like this:
# 1. importing libraries
# 2. loading data
# 3. preprocessing data
# 4. converting categorical labels to numerical labels
# 5. Dividing data into independent and dependent variables
# 6. splitting data into training and testing sets
# 7. Scaling the data
# 8. training the model
# 9. evaluating the model
# 10. saving the model
# With Pipeline, we can combine steps 3 to 8 into a single pipeline object, which makes our code cleaner and more efficient. We can also easily add or remove steps from the pipeline as needed.
# For example, we can create a pipeline that includes text preprocessing, vectorization, and model training all in one step. This way, we can fit the pipeline to our training data and make predictions on our test data without having to manually execute each step separately.

In [4]:
# Step 2: Load the Dataset

DATA_PATH = Path("Restaurant_Reviews.tsv")

dataset = pd.read_csv(DATA_PATH, sep="\t")
dataset.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [5]:
print("Dataset shape:", dataset.shape)

Dataset shape: (1000, 2)


In [6]:
print("Columns and data types:")
print(dataset.dtypes)

Columns and data types:
Review      str
Liked     int64
dtype: object


In [7]:
print("Missing values:")
print(dataset.isnull().sum())

Missing values:
Review    0
Liked     0
dtype: int64


In [8]:
# Step 3: Exploratory Checks
# Before modeling, we check class balance and basic text length statistics. These values are also useful later as a monitoring baseline.

print(dataset["Liked"].value_counts())

Liked
1    500
0    500
Name: count, dtype: int64


In [9]:
class_balance = dataset["Liked"].value_counts().rename_axis("Liked").reset_index(name="count")
print(class_balance)


   Liked  count
0      1    500
1      0    500


In [10]:
class_balance["percentage"] = (class_balance["count"] / len(dataset) * 100).round(2)
class_balance

,Liked,count,percentage
0,1,500,50.0
1,0,500,50.0


In [11]:
dataset["review_length"] = dataset["Review"].astype(str).str.split().str.len() # Calculate the number of words in each review
dataset[["Review", "review_length"]].head()


,Review,review_length
0,Wow... Loved this place.,4
1,Crust is not good.,4
2,Not tasty and the texture was just nasty.,8
3,Stopped by during the late May bank holiday of...,15
4,The selection on the menu was great and so wer...,12


In [12]:
# To display entire review text without truncation, we can set the pandas display option for maximum column width. This allows us to see the full content of the 'Review' column without it being cut off.

pd.set_option("display.max_colwidth", None) # Set max column width to None to display full review text

dataset[["Review", "review_length"]].head() # Display the first few reviews along with their lengths

,Review,review_length
0,Wow... Loved this place.,4
1,Crust is not good.,4
2,Not tasty and the texture was just nasty.,8
3,Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.,15
4,The selection on the menu was great and so were the prices.,12


In [13]:
length_summary = dataset["review_length"].describe().to_frame("review_length")
length_summary

,review_length
count,1000.000000
mean,10.894000
std,6.257469
min,1.000000
25%,6.000000
50%,10.000000
75%,15.000000
max,32.000000


In [14]:
# Step 4: Text Preprocessing Function
# A common mistake in sentiment analysis is removing negation words. For example, not good and good should not be treated as the same message. The function below removes punctuation and common stopwords while preserving important negations.

CUSTOM_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "while", "is", "am", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "for", "with", "at", "by", "from", "this", "that", "these", "those", "it", "its",
    "as", "so", "very", "just", "than", "then", "there", "here", "i", "we", "you", "he", "she", "they", "them",
    "my", "our", "your", "his", "her", "their", "me", "us", "do", "does", "did", "have", "has", "had"
}
NEGATION_WORDS = {"no", "not", "nor", "never", "none", "cannot", "can't", "dont", "don't", "didnt", "didn't", "isnt", "isn't", "wasnt", "wasn't", "wont", "won't"}
STOPWORDS = CUSTOM_STOPWORDS - NEGATION_WORDS


def clean_review(text: str) -> str: # clean_review("This restaurant is not good at all!") -> "restaurant not good"
    text = str(text).lower() # Convert to lowercase. text = "This restaurant is not good at all!" -> "this restaurant is not good at all!"
    text = re.sub(r"<.*?>", " ", text) # Remove HTML tags. text = "this restaurant is not good at all!" -> "this restaurant is not good at all!"
    text = re.sub(r"[^a-zA-Z' ]", " ", text) # Remove punctuation except for apostrophes. text = "this restaurant is not good at all!" -> "this restaurant is not good at all"
    tokens = [token.strip(string.punctuation) for token in text.split()] # Split into tokens and remove leading/trailing punctuation. text = "this restaurant is not good at all" -> ["this", "restaurant", "is", "not", "good", "at", "all"]
    tokens = [token for token in tokens if token and token not in STOPWORDS and len(token) > 1] # Remove stopwords and single-character tokens. text = ["this", "restaurant", "is", "not", "good", "at", "all"] -> ["restaurant", "not", "good"]
    return " ".join(tokens) # Join tokens back into a single string. text = ["restaurant", "not", "good"] -> "restaurant not good"


dataset["clean_review"] = dataset["Review"].apply(clean_review)
dataset[["Review", "clean_review", "Liked"]].head(10)

,Review,clean_review,Liked
0,Wow... Loved this place.,wow loved place,1
1,Crust is not good.,crust not good,0
2,Not tasty and the texture was just nasty.,not tasty texture nasty,0
3,Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.,stopped during late may bank holiday off rick steve recommendation loved,1
4,The selection on the menu was great and so were the prices.,selection menu great prices,1
5,Now I am getting angry and I want my damn pho.,now getting angry want damn pho,0
6,Honeslty it didn't taste THAT fresh.),honeslty didn't taste fresh,0
7,The potatoes were like rubber and you could tell they had been made up ahead of time being kept under a warmer.,potatoes like rubber could tell made up ahead time kept under warmer,0
8,The fries were great too.,fries great too,1
9,A great touch.,great touch,1


In [15]:
# Step 5: Train-Test Split
# We use a stratified split so the positive and negative classes remain balanced in both training and testing data.

# What is stratified split?
# A stratified split is a method of splitting a dataset into training and testing sets while preserving the proportion of classes in the target variable. This is particularly important in classification problems where the classes may be imbalanced. By using a stratified split, we ensure that both the training and testing sets have a representative distribution of the classes, which can lead to better model performance and more reliable evaluation metrics.
# In a stratified split, the dataset is divided in such a way that the percentage of samples for each class in the target variable is the same in both the training and testing sets as it is in the original dataset. This helps to ensure that the model is trained and evaluated on data that is representative of the overall distribution of classes, which can improve the generalization of the model to unseen data. In scikit-learn, you can perform a stratified split using the `train_test_split` function with the `stratify` parameter set to the target variable.
# eg: If you have a column named "Education" with classes "High School", "Bachelor's", and "Master's", and you want to split your dataset into training and testing sets while preserving the class distribution, then you can go for Stratified Split. This will ensure that the proportion of "High School", "Bachelor's", and "Master's" is the same in both the training and testing sets as it is in the original dataset. This is important to ensure that your model is trained and evaluated on data that is representative of the overall distribution of classes, which can lead to better performance and more reliable evaluation metrics.

X = dataset["Review"]
y = dataset["Liked"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
print("Train target balance:")
print(y_train.value_counts(normalize=True).round(2))

Training records: 800
Testing records: 200
Train target balance:
Liked
1    0.5
0    0.5
Name: proportion, dtype: float64


## Step 6: Build the NLP Model Pipeline

The pipeline keeps preprocessing, vectorization, and classification together. This is important for deployment because the same transformations used during training must be used during prediction.

In [16]:
sentiment_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(preprocessor=clean_review, max_features=1500, ngram_range=(1, 2))),
    ("model", LogisticRegression(max_iter=1000, solver="liblinear", random_state=42))
])
# ngram_range=(1, 2) means we are using both unigrams (single words) and bigrams (pairs of consecutive words) as features in our model. This allows the model to capture more context and potentially improve its ability to understand the sentiment of the reviews. For example, the bigram "not good" can be an important feature for identifying negative sentiment, which might be missed if we only used unigrams.
# eg: sent = "This restaurant is not good at all!" 
# -> unigrams: ["this", "restaurant", "is", "not", "good", "at", "all"] 
# -> bigrams: ["this restaurant", "restaurant is", "is not", "not good", "good at", "at all"] 
# -> features: ["this", "restaurant", "is", "not", "good", "at", "all", "this restaurant", "restaurant is", "is not", "not good", "good at", "at all"]

sentiment_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function cle...002353E599D00>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [17]:
# Step 7: Model Evaluation
# Accuracy is useful, but for classification monitoring we should also track precision, recall, F1 score, and confusion matrix values.

y_pred = sentiment_pipeline.predict(X_test)
y_prob = sentiment_pipeline.predict_proba(X_test).max(axis=1)
# predict() - gives the predicted class labels for the input data.
# predict_proba() - gives the predicted probabilities for each class
# eg: In restaurant review sentiment analysis, if we have a review:
# "The food was great but the service was terrible", 
# the model might predict a positive sentiment with a probability of 0.7 and a negative sentiment with a probability of 0.3. 
# In this case, predict() would return the class label "positive" (assuming it's the class with the highest probability), 
# while predict_proba() would return [0.3, 0.7] indicating the probabilities for both classes.

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "Average Confidence": float(y_prob.mean())
}

pd.DataFrame([metrics]).round(4)

,Accuracy,Precision,Recall,F1 Score,Average Confidence
0,0.795,0.7706,0.84,0.8038,0.7517


In [18]:
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)
confusion

              precision    recall  f1-score   support

    Negative       0.82      0.75      0.79       100
    Positive       0.77      0.84      0.80       100

    accuracy                           0.80       200
   macro avg       0.80      0.79      0.79       200
weighted avg       0.80      0.80      0.79       200



,Predicted Negative,Predicted Positive
Actual Negative,75,25
Actual Positive,16,84


In [19]:
# Step 8: Error and Confidence Analysis
# A deployment should not only return predictions. It should also flag uncertain predictions and help the business understand where the model may fail.

evaluation_df = pd.DataFrame({
    "review": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred,
    "confidence": y_prob
})
evaluation_df.head()


,review,actual,predicted,confidence
0,Awesome selection of beer.,1,1,0.877482
1,"Not much flavor to them, and very poorly constructed.",0,0,0.957109
2,"Worse of all, he humiliated his worker right in front of me..Bunch of horrible name callings.",0,0,0.566447
3,"Host staff were, for lack of a better word, BITCHES!",0,1,0.524904
4,Great brunch spot.,1,1,0.955106


In [20]:
evaluation_df["correct"] = evaluation_df["actual"] == evaluation_df["predicted"]
evaluation_df.head()

,review,actual,predicted,confidence,correct
0,Awesome selection of beer.,1,1,0.877482,True
1,"Not much flavor to them, and very poorly constructed.",0,0,0.957109,True
2,"Worse of all, he humiliated his worker right in front of me..Bunch of horrible name callings.",0,0,0.566447,True
3,"Host staff were, for lack of a better word, BITCHES!",0,1,0.524904,False
4,Great brunch spot.,1,1,0.955106,True


In [21]:
evaluation_df["actual_label"] = evaluation_df["actual"].map({0: "Negative", 1: "Positive"})
evaluation_df["predicted_label"] = evaluation_df["predicted"].map({0: "Negative", 1: "Positive"})
evaluation_df.head()


,review,actual,predicted,confidence,correct,actual_label,predicted_label
0,Awesome selection of beer.,1,1,0.877482,True,Positive,Positive
1,"Not much flavor to them, and very poorly constructed.",0,0,0.957109,True,Negative,Negative
2,"Worse of all, he humiliated his worker right in front of me..Bunch of horrible name callings.",0,0,0.566447,True,Negative,Negative
3,"Host staff were, for lack of a better word, BITCHES!",0,1,0.524904,False,Negative,Positive
4,Great brunch spot.,1,1,0.955106,True,Positive,Positive


In [22]:
low_confidence_cases = evaluation_df.sort_values("confidence").head(10) # Get the 10 cases with the lowest confidence
low_confidence_cases[["review", "actual_label", "predicted_label", "confidence", "correct"]]

,review,actual_label,predicted_label,confidence,correct
32,You can't beat that.,Positive,Positive,0.500376,True
80,"If you look for authentic Thai food, go else where.",Negative,Positive,0.503567,False
48,So they performed.,Positive,Negative,0.504619,False
54,I was so insulted.,Negative,Negative,0.504619,True
171,"I had high hopes for this place since the burgers are cooked over a charcoal grill, but unfortunately the taste fell flat, way flat.",Negative,Positive,0.512190,False
132,This place deserves one star and 90% has to do with the food.,Negative,Positive,0.512826,False
36,This place lacked style!!,Negative,Negative,0.513196,True
131,"Overall, I like there food and the service.",Positive,Positive,0.513673,True
35,A lady at the table next to us found a live green caterpillar In her salad.,Negative,Negative,0.516594,True
173,The warm beer didn't help.,Negative,Positive,0.516975,False


In [23]:
mistakes = evaluation_df[evaluation_df["correct"] == False].sort_values("confidence", ascending=False)
mistakes[["review", "actual_label", "predicted_label", "confidence"]].head(10)

,review,actual_label,predicted_label,confidence
182,We made the drive all the way from North Scottsdale... and I was not one bit disappointed!,Positive,Negative,0.988335
113,"Not to mention the combination of pears, almonds and bacon is a big winner!",Positive,Negative,0.855002
56,Never had anything to complain about here.,Positive,Negative,0.832606
118,"I do love sushi, but I found Kabuki to be over-priced, over-hip and under-services.",Negative,Positive,0.812800
84,Some may say this buffet is pricey but I think you get what you pay for and this place you are getting quite a lot!,Positive,Negative,0.785468
46,It also took her forever to bring us the check when we asked for it.,Negative,Positive,0.748817
34,"This really is how Vegas fine dining used to be, right down to the menus handed to the ladies that have no prices listed.",Positive,Negative,0.726406
133,"The descriptions said ""yum yum sauce"" and another said ""eel sauce"", yet another said ""spicy mayo""...well NONE of the rolls had sauces on them.",Negative,Positive,0.713305
142,"As for the ""mains,"" also uninspired.",Negative,Positive,0.696992
7,"Service is quick and even ""to go"" orders are just like we like it!",Positive,Negative,0.672854


## Step 9: Explain Important Words

For Logistic Regression, positive coefficients push predictions toward positive sentiment and negative coefficients push predictions toward negative sentiment.

In [24]:
sentiment_pipeline.named_steps["vectorizer"]

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function cle...002353E599D00>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [25]:
sentiment_pipeline.named_steps["model"]

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [26]:
sentiment_pipeline.named_steps["vectorizer"].get_feature_names_out()

array(['about', 'about place', 'above', ..., 'yummy tummy', 'zero',
       'zero stars'], dtype=object)

In [27]:
vectorizer = sentiment_pipeline.named_steps["vectorizer"]
model = sentiment_pipeline.named_steps["model"]
feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

importance_df = pd.DataFrame({"term": feature_names, "coefficient": coefs})
positive_terms = importance_df.sort_values("coefficient", ascending=False).head(15)
negative_terms = importance_df.sort_values("coefficient", ascending=True).head(15)

print("Top positive terms")
display(positive_terms)
print("Top negative terms")
display(negative_terms)

Top positive terms


,term,coefficient
342,great,2.309639
330,good,1.865091
177,delicious,1.652628
263,fantastic,1.468294
303,friendly,1.334609
18,amazing,1.313928
46,awesome,1.302684
440,love,1.239653
1432,won disappointed,1.195411
539,nice,1.156303


Top negative terms


,term,coefficient
591,not,-1.837605
57,bad,-1.432570
1450,worst,-1.395855
1354,wasn,-1.319562
615,not good,-1.295944
566,no,-1.287322
1280,terrible,-1.264035
200,don,-1.253880
43,avoid,-1.219043
90,bland,-1.132601


# Monitoring Section in Notebook

Monitoring is the difference between a notebook model and a usable ML solution. In a real system, monitoring answers questions such as:

- Are inputs similar to the data used for training?
- Are predictions becoming too uncertain?
- Is the positive/negative prediction mix changing sharply?
- Which predictions should be sent to a human reviewer?
- What examples should be collected for future retraining?

## Step 10: Create a Monitoring Baseline

The baseline records normal behavior from the training data. Live predictions can then be compared against this reference.

In [28]:
# training_baseline is a dictionary that captures key statistics about the training data and model performance. This can be used as a reference point for monitoring the model in production. If future data or model updates deviate significantly from this baseline, it may indicate issues that need to be investigated.
training_baseline = {
    "train_rows": len(dataset), # The number of records in the training dataset. This helps us monitor if there are any significant changes in the amount of data being used for training, which could affect model performance.
    "train_positive_rate": float(dataset["Liked"].mean()), # The average value of the target variable in the training dataset. This helps us monitor if there are any significant changes in the class distribution of the training data, which could affect model performance.
    "train_avg_review_length": float(dataset["review_length"].mean()), # The average length of reviews in the training dataset. This helps us monitor if there are any significant changes in the nature of the reviews being used for training, which could affect model performance.
    "train_p95_review_length": float(dataset["review_length"].quantile(0.95)), # The 95th percentile of review lengths in the training dataset. This helps us monitor if there are any significant changes in the distribution of review lengths, which could affect model performance.
    "test_accuracy": metrics["Accuracy"], # The accuracy of the model on the test set. This serves as a baseline for monitoring future model performance in production.
    "test_f1": metrics["F1 Score"], # The F1 score of the model on the test set. This serves as another baseline for monitoring future model performance in production, especially in cases where there is class imbalance.
    "test_avg_confidence": metrics["Average Confidence"] # The average confidence of the model's predictions on the test set. This serves as a baseline for monitoring future model confidence in production, which can help identify if the model is becoming less certain about its predictions over time.
}

pd.DataFrame([training_baseline]).round(4)

,train_rows,train_positive_rate,train_avg_review_length,train_p95_review_length,test_accuracy,test_f1,test_avg_confidence
0,1000,0.5,10.894,23.0,0.795,0.8038,0.7517


## Step 11: Prediction Function with Logging

This function returns predictions and creates the type of log that can feed a monitoring dashboard.

In [29]:
def predict_with_monitoring(reviews, pipeline=sentiment_pipeline, confidence_threshold=0.65):
    probabilities = pipeline.predict_proba(reviews)
    predictions = pipeline.predict(reviews)
    rows = []
    for review, pred, prob in zip(reviews, predictions, probabilities):
        confidence = float(prob[pred])
        rows.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "review": review,
            "clean_review": clean_review(review),
            "prediction": "Positive" if pred == 1 else "Negative",
            "predicted_class": int(pred),
            "confidence": round(confidence, 4),
            "review_length": len(str(review).split()),
            "needs_human_review": confidence < confidence_threshold
        })
    return pd.DataFrame(rows)

sample_live_reviews = [
    "The pasta was outstanding and the staff made us feel welcome.",
    "Food was not good and the table was dirty.",
    "Average place. Nothing terrible, nothing special.",
    "The biryani was excellent but delivery was late.",
    "Worst service ever. Never coming again."
]

prediction_log = predict_with_monitoring(sample_live_reviews)
prediction_log

,timestamp,review,clean_review,prediction,predicted_class,confidence,review_length,needs_human_review
0,2026-05-26 12:23:26,The pasta was outstanding and the staff made us feel welcome.,pasta outstanding staff made feel welcome,Positive,1,0.7030,11,False
1,2026-05-26 12:23:26,Food was not good and the table was dirty.,food not good table dirty,Negative,0,0.8548,9,False
2,2026-05-26 12:23:26,"Average place. Nothing terrible, nothing special.",average place nothing terrible nothing special,Negative,0,0.9690,6,False
3,2026-05-26 12:23:26,The biryani was excellent but delivery was late.,biryani excellent delivery late,Positive,1,0.7575,8,False
4,2026-05-26 12:23:26,Worst service ever. Never coming again.,worst service ever never coming again,Negative,0,0.9699,6,False


## Step 12: Notebook Monitoring Dashboard

The dashboard below uses simple pandas tables so it runs even in lightweight environments. These same ideas are added to Streamlit later with interactive widgets and charts.

In [30]:
def build_monitoring_summary(log_df, baseline):
    if log_df.empty:
        return pd.DataFrame()

    live_positive_rate = log_df["predicted_class"].mean()
    low_confidence_rate = log_df["needs_human_review"].mean()
    avg_live_length = log_df["review_length"].mean()
    # We can set a threshold for input length drift. For example, if the average review length in live data deviates by more than 50% from the training average, we can flag it as a potential drift.
    input_length_drift = abs(avg_live_length - baseline["train_avg_review_length"]) > 0.5 * baseline["train_avg_review_length"]

    summary = {
        "live_prediction_count": len(log_df),
        "live_positive_rate": live_positive_rate,
        "training_positive_rate": baseline["train_positive_rate"],
        "low_confidence_rate": low_confidence_rate,
        "avg_live_review_length": avg_live_length,
        "avg_training_review_length": baseline["train_avg_review_length"],
        "input_length_drift_flag": input_length_drift,
        "human_review_required_count": int(log_df["needs_human_review"].sum())
    }
    return pd.DataFrame([summary])

monitoring_summary = build_monitoring_summary(prediction_log, training_baseline)
monitoring_summary.round(4)

,live_prediction_count,live_positive_rate,training_positive_rate,low_confidence_rate,avg_live_review_length,avg_training_review_length,input_length_drift_flag,human_review_required_count
0,5,0.4,0.5,0.0,8.0,10.894,False,0


## Step 14: Save the Model Artifact

The saved artifact contains both the vectorizer and model, which keeps deployment consistent with training.

In [31]:
MODEL_PATH = Path("restaurant_sentiment_pipeline.joblib")
joblib.dump(sentiment_pipeline, MODEL_PATH)
print(f"Saved model pipeline to: {MODEL_PATH.resolve()}")

Saved model pipeline to: C:\Users\hp\Desktop\NLP Project\restaurant_sentiment_pipeline.joblib


# Streamlit Deployment Section

The Streamlit app adds a usable interface for:

- Single review prediction
- Batch CSV/TSV scoring
- Live prediction logging
- Model quality metrics
- Monitoring cards
- Prediction mix chart
- Confidence trend chart
- Human-review flags for uncertain predictions

The app file is included with this notebook as `streamlit_restaurant_monitoring_app.py`.

## Step 15: Streamlit App Code

The cell below writes the Streamlit application to disk. It uses the same preprocessing, model training, prediction logging, and monitoring rules that we developed in the notebook.

In [33]:
app_code = r"""import re
import string
from collections import Counter
from datetime import datetime
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

DATA_PATH = Path(__file__).with_name("Restaurant_Reviews.tsv")
MODEL_PATH = Path(__file__).with_name("restaurant_sentiment_pipeline.joblib")

CUSTOM_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "while", "is", "am", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "for", "with", "at", "by", "from", "this", "that", "these", "those", "it", "its",
    "as", "so", "very", "just", "than", "then", "there", "here", "i", "we", "you", "he", "she", "they", "them",
    "my", "our", "your", "his", "her", "their", "me", "us", "do", "does", "did", "have", "has", "had"
}
NEGATION_WORDS = {"no", "not", "nor", "never", "none", "cannot", "can't", "dont", "don't", "didnt", "didn't", "isnt", "isn't", "wasnt", "wasn't", "wont", "won't"}
STOPWORDS = CUSTOM_STOPWORDS - NEGATION_WORDS


def clean_review(text: str) -> str:
    \"\"\"Clean review text while preserving negation words that matter for sentiment.\"\"\"
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z' ]", " ", text)
    tokens = [token.strip(string.punctuation) for token in text.split()]
    tokens = [token for token in tokens if token and token not in STOPWORDS and len(token) > 1]
    return " ".join(tokens)


@st.cache_data
def load_data() -> pd.DataFrame:
    data = pd.read_csv(DATA_PATH, sep="\t")
    data["clean_review"] = data["Review"].apply(clean_review)
    data["review_length"] = data["Review"].astype(str).str.split().str.len()
    return data


@st.cache_resource
def train_or_load_pipeline():
    data = load_data()
    X_train, X_test, y_train, y_test = train_test_split(
        data["Review"], data["Liked"], test_size=0.2, random_state=42, stratify=data["Liked"]
    )
    pipeline = Pipeline([
        ("vectorizer", CountVectorizer(preprocessor=clean_review, max_features=1500, ngram_range=(1, 2))),
        ("model", LogisticRegression(max_iter=1000, solver="liblinear", random_state=42))
    ])
    pipeline.fit(X_train, y_train)
    joblib.dump(pipeline, MODEL_PATH)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test).max(axis=1)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "Avg Confidence": float(y_prob.mean()),
        "False Positives": int(fp),
        "False Negatives": int(fn),
    }
    baseline = {
        "train_avg_length": float(data["review_length"].mean()),
        "train_p95_length": float(data["review_length"].quantile(0.95)),
        "positive_rate": float(data["Liked"].mean()),
    }
    return pipeline, metrics, baseline


def prediction_frame(pipeline, reviews):
    probabilities = pipeline.predict_proba(reviews)
    predictions = pipeline.predict(reviews)
    rows = []
    for review, pred, prob in zip(reviews, predictions, probabilities):
        confidence = float(prob[pred])
        rows.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "review": review,
            "clean_review": clean_review(review),
            "prediction": "Positive" if pred == 1 else "Negative",
            "predicted_class": int(pred),
            "confidence": round(confidence, 4),
            "review_length": len(str(review).split()),
            "needs_human_review": confidence < 0.65,
        })
    return pd.DataFrame(rows)


def top_terms(texts, n=12):
    tokens = " ".join(clean_review(text) for text in texts).split()
    return pd.DataFrame(Counter(tokens).most_common(n), columns=["term", "count"])


def show_metric_cards(metrics, baseline):
    cols = st.columns(4)
    cols[0].metric("Model accuracy", f"{metrics['Accuracy']:.1%}")
    cols[1].metric("F1 score", f"{metrics['F1 Score']:.1%}")
    cols[2].metric("Average confidence", f"{metrics['Avg Confidence']:.1%}")
    cols[3].metric("Training positive rate", f"{baseline['positive_rate']:.1%}")


def monitoring_dashboard(log_df: pd.DataFrame, baseline: dict):
    if log_df.empty:
        st.info("No live predictions yet. Enter a review or upload a small batch to populate the monitoring dashboard.")
        return

    positive_rate = log_df["predicted_class"].mean()
    low_conf_rate = log_df["needs_human_review"].mean()
    avg_length = log_df["review_length"].mean()
    drift_flag = abs(avg_length - baseline["train_avg_length"]) > 0.5 * baseline["train_avg_length"]

    cols = st.columns(4)
    cols[0].metric("Live predictions", len(log_df))
    cols[1].metric("Live positive rate", f"{positive_rate:.1%}")
    cols[2].metric("Low confidence rate", f"{low_conf_rate:.1%}")
    cols[3].metric("Avg review length", f"{avg_length:.1f} words")

    if drift_flag:
        st.warning("Input length drift detected: recent reviews are much shorter or longer than the training baseline.")
    if low_conf_rate > 0.25:
        st.warning("More than 25% of recent predictions need human review. This is a useful retraining or threshold-tuning signal.")

    left, right = st.columns(2)
    with left:
        st.subheader("Prediction mix")
        mix = log_df["prediction"].value_counts().rename_axis("sentiment").reset_index(name="count")
        st.bar_chart(mix, x="sentiment", y="count")
    with right:
        st.subheader("Confidence over time")
        confidence_trend = log_df.reset_index().rename(columns={"index": "request_number"})[["request_number", "confidence"]]
        st.line_chart(confidence_trend, x="request_number", y="confidence")

    st.subheader("Recent prediction log")
    st.dataframe(log_df.tail(20).sort_index(ascending=False), use_container_width=True, hide_index=True)

    st.subheader("Top words in recent traffic")
    st.dataframe(top_terms(log_df["review"].tail(50)), use_container_width=True, hide_index=True)


def main():
    st.set_page_config(page_title="Restaurant Review Sentiment Monitor", page_icon=":knife_fork_plate:", layout="wide")
    st.title("Restaurant Review Sentiment Monitor")
    st.caption("A deployment-style Streamlit app with live prediction logging, quality checks, and monitoring signals.")

    if not DATA_PATH.exists():
        st.error(f"Dataset not found at {DATA_PATH}. Keep Restaurant_Reviews.tsv in the same folder as this app.")
        st.stop()

    pipeline, model_metrics, training_baseline = train_or_load_pipeline()
    show_metric_cards(model_metrics, training_baseline)

    if "prediction_log" not in st.session_state:
        st.session_state.prediction_log = pd.DataFrame(columns=[
            "timestamp", "review", "clean_review", "prediction", "predicted_class", "confidence", "review_length", "needs_human_review"
        ])

    st.divider()
    left_panel, right_panel = st.columns([1, 1])

    with left_panel:
        st.subheader("Single review prediction")
        sample_review = st.text_area("Customer review", "The food was delicious, but the service was very slow.", height=120)
        if st.button("Predict review", type="primary"):
            result = prediction_frame(pipeline, [sample_review])
            st.session_state.prediction_log = pd.concat([st.session_state.prediction_log, result], ignore_index=True)
            label = result.loc[0, "prediction"]
            confidence = result.loc[0, "confidence"]
            st.success(f"Prediction: {label} with {confidence:.1%} confidence")
            if result.loc[0, "needs_human_review"]:
                st.warning("Confidence is below 65%, so this prediction should be reviewed by a human.")

    with right_panel:
        st.subheader("Batch scoring")
        uploaded = st.file_uploader("Upload CSV/TSV with a Review column", type=["csv", "tsv"])
        if uploaded is not None:
            sep = "\t" if uploaded.name.endswith(".tsv") else ","
            batch = pd.read_csv(uploaded, sep=sep)
            if "Review" not in batch.columns:
                st.error("The uploaded file must contain a column named Review.")
            else:
                scored = prediction_frame(pipeline, batch["Review"].astype(str).tolist())
                st.session_state.prediction_log = pd.concat([st.session_state.prediction_log, scored], ignore_index=True)
                st.dataframe(scored, use_container_width=True, hide_index=True)
                st.download_button(
                    "Download scored predictions",
                    scored.to_csv(index=False).encode("utf-8"),
                    "scored_restaurant_reviews.csv",
                    "text/csv"
                )

    st.divider()
    st.header("Monitoring dashboard")
    monitoring_dashboard(st.session_state.prediction_log, training_baseline)

    with st.expander("Model and monitoring notes"):
        st.write(
            "This app retrains a compact logistic-regression pipeline on startup, saves a joblib artifact, "
            "and tracks live predictions in the Streamlit session. In production, the prediction log would be "
            "written to a database or observability platform, and the drift checks would compare live traffic "
            "against a stable training baseline."
        )


if __name__ == "__main__":
    main()"""

Path("streamlit_restaurant_monitoring_app.py").write_text(app_code)
print("Streamlit app created: streamlit_restaurant_monitoring_app.py")

Streamlit app created: streamlit_restaurant_monitoring_app.py


## Step 16: How to Run the Streamlit App

Run the command below from the folder containing this notebook and the TSV file:

```bash
streamlit run streamlit_restaurant_monitoring_app.py
```

The app will open in the browser. Each prediction made in the app is added to a session-level monitoring log, which updates the dashboard immediately.

In [36]:
print("Run this command in a terminal:")
print("streamlit run streamlit_restaurant_monitoring_app.py")
#or
print("python -m streamlit run streamlit_restaurant_monitoring_app.py")

Run this command in a terminal:
streamlit run streamlit_restaurant_monitoring_app.py
python -m streamlit run streamlit_restaurant_monitoring_app.py


## Step 17: Production Monitoring Recommendations

For real production use, the monitoring layer should be connected to persistent storage and alerting.

Recommended additions:

1. Store every prediction request, response, confidence score, and timestamp in a database.
2. Capture actual user feedback later, so real accuracy can be measured after deployment.
3. Track daily positive rate, low-confidence rate, average review length, empty inputs, and top new words.
4. Create alerts when low-confidence predictions cross a threshold.
5. Trigger retraining when the model sees new restaurant vocabulary, menu items, or review patterns.
6. Keep a human-review workflow for low-confidence or high-impact reviews.

## Conclusion

In this notebook, we moved from NLP preprocessing to a deployment-oriented machine learning workflow. We trained a restaurant review sentiment model, evaluated it using classification metrics, added notebook-based monitoring, saved the pipeline, and created a Streamlit app with live prediction and dashboard functionality.

The final solution is useful for teaching because it shows the full journey: data understanding, preprocessing, model building, evaluation, monitoring, and deployment. It is also practical because the same monitoring ideas used in the notebook are available in the Streamlit app for real-time review analysis.

In [35]:
# Exporting the requirements.txt file for Github:
# execute this code in the terminal to create a requirements.txt file with all the dependencies used in this project. This is important for sharing the project on GitHub or deploying it, as it allows others to easily install the necessary libraries.
# pip freeze > requirements.txt
print("Exported requirements.txt for GitHub.")

Exported requirements.txt for GitHub.
